In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 01: revenue_kpis
-- ══════════════════════════════════════════════════════
-- Season-level scorecard for the top of the dashboard.

SELECT
  -- Total season gross revenue
  ROUND(SUM(gross_revenue), 0)                  AS total_gross_revenue,

  -- Total net revenue after all discounts
  ROUND(SUM(net_revenue), 0)                    AS total_net_revenue,

  -- Total discounts given away — the cost of the slump promo strategy
  ROUND(SUM(total_discount_given), 0)           AS total_discounts_given,

  -- Discount as % of gross — how much margin we gave away
  ROUND(
    SUM(total_discount_given) * 100.0
    / NULLIF(SUM(gross_revenue), 0), 1
  )                                             AS discount_as_pct_of_gross,

  -- Average season fill rate
  ROUND(AVG(arena_fill_pct) * 100, 1)          AS avg_fill_rate_pct,

  -- Jersey Night revenue (single game for context)
  ROUND(
    MAX(CASE WHEN is_jersey_night = TRUE THEN gross_revenue ELSE 0 END), 0
  )                                             AS jersey_night_revenue,

  -- Total games played
  COUNT(*)                                      AS total_games,

  -- Sellout count
  SUM(CASE WHEN arena_fill_pct >= 0.99 THEN 1 ELSE 0 END) AS sellouts

FROM icebase.gold.game_revenue;

In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 02: revenue_by_game
-- ══════════════════════════════════════════════════════
-- Per-game revenue trend — the main line chart showing the season arc.
-- Gross and net revenue both plotted so discount impact is visible.

SELECT
  game_number,
  game_date,
  opponent,
  season_phase,
  is_jersey_night,
  ROUND(gross_revenue, 0)                       AS gross_revenue,
  ROUND(net_revenue, 0)                         AS net_revenue,
  ROUND(total_discount_given, 0)                AS discounts_given,
  ROUND(arena_fill_pct * 100, 1)               AS fill_rate_pct,
  ROUND(revenue_index, 3)                       AS revenue_index,
  ROUND(cumulative_revenue, 0)                  AS cumulative_revenue,
  result
FROM icebase.gold.game_revenue
WHERE is_home_game = TRUE
ORDER BY game_number;

In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 03: fill_rate_trend
-- ══════════════════════════════════════════════════════
-- Fill rate per game with 3-game rolling average.
-- Highlights the slump drop and jersey night spike clearly.

SELECT
  game_number,
  game_date,
  season_phase,
  is_jersey_night,
  ROUND(arena_fill_pct * 100, 1)               AS fill_rate_pct,
  ROUND(fill_rate_3game_avg * 100, 1)          AS fill_rate_3game_avg_pct,
  -- Reference line: 60% threshold (alert trigger)
  60.0                                          AS alert_threshold_pct
FROM icebase.gold.game_revenue
WHERE is_home_game = TRUE
ORDER BY game_number;

In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 04: revenue_by_phase
-- ══════════════════════════════════════════════════════
-- Aggregated revenue by season phase — bar chart showing the full arc.

SELECT
  season_phase,
  COUNT(*)                                      AS games,
  ROUND(SUM(gross_revenue), 0)                 AS total_gross_revenue,
  ROUND(SUM(net_revenue), 0)                   AS total_net_revenue,
  ROUND(SUM(total_discount_given), 0)          AS total_discounts,
  ROUND(AVG(gross_revenue), 0)                 AS avg_revenue_per_game,
  ROUND(AVG(arena_fill_pct) * 100, 1)         AS avg_fill_rate_pct,
  ROUND(AVG(promo_rate) * 100, 1)             AS avg_promo_rate_pct,
  CASE season_phase
    WHEN 'hot_start'    THEN 1
    WHEN 'slump'        THEN 2
    WHEN 'late_push'    THEN 3
    WHEN 'bridge'       THEN 4
    WHEN 'jersey_night' THEN 5
    WHEN 'post_jersey'  THEN 6
    ELSE 7
  END                                          AS phase_order
FROM icebase.gold.game_revenue
WHERE is_home_game = TRUE
GROUP BY season_phase
ORDER BY phase_order;

In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 05: promo_impact_by_phase
-- ══════════════════════════════════════════════════════
-- The "bad story" in numbers — discount cost breakdown by phase.
-- This shows exactly how much the slump promo strategy cost.

SELECT
  season_phase,
  COUNT(*)                                      AS games,
  SUM(promo_tickets)                           AS total_promo_tickets,
  ROUND(AVG(promo_rate) * 100, 1)             AS avg_promo_rate_pct,
  ROUND(SUM(total_discount_given), 0)          AS total_discount_cost,
  ROUND(AVG(avg_discount_pct), 1)             AS avg_discount_depth_pct,
  -- Revenue recovered vs what we could have made at full price
  ROUND(SUM(gross_revenue), 0)                AS actual_gross_revenue,
  ROUND(SUM(gross_revenue) + SUM(total_discount_given), 0)
                                               AS full_price_revenue_estimate,
  CASE season_phase
    WHEN 'hot_start'    THEN 1
    WHEN 'slump'        THEN 2
    WHEN 'late_push'    THEN 3
    WHEN 'bridge'       THEN 4
    WHEN 'jersey_night' THEN 5
    WHEN 'post_jersey'  THEN 6
    ELSE 7
  END                                          AS phase_order
FROM icebase.gold.game_revenue
WHERE is_home_game = TRUE
GROUP BY season_phase
ORDER BY phase_order;

In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 06: top_revenue_games
-- ══════════════════════════════════════════════════════
-- Best and worst revenue games — useful table widget.

SELECT
  game_number,
  game_date,
  opponent,
  season_phase,
  is_jersey_night,
  result,
  ROUND(gross_revenue, 0)                       AS gross_revenue,
  ROUND(net_revenue, 0)                         AS net_revenue,
  ROUND(arena_fill_pct * 100, 1)               AS fill_rate_pct,
  ROUND(promo_rate * 100, 1)                   AS promo_rate_pct,
  ROUND(revenue_index, 2)                       AS revenue_index
FROM icebase.gold.game_revenue
WHERE is_home_game = TRUE
ORDER BY gross_revenue DESC;

In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 07: section_tier_revenue
-- ══════════════════════════════════════════════════════
-- Revenue breakdown by seat tier — shows premium vs general seating mix.

SELECT
  t.section_tier,
  t.seat_tier_rank,
  COUNT(*)                                      AS tickets_sold,
  ROUND(SUM(t.ticket_price), 0)                AS total_revenue,
  ROUND(AVG(t.ticket_price), 2)                AS avg_ticket_price,
  ROUND(
    SUM(CASE WHEN t.is_promo_ticket = TRUE THEN 1 ELSE 0 END)
    * 100.0 / COUNT(*), 1
  )                                             AS promo_rate_pct,
  ROUND(
    SUM(t.ticket_price) * 100.0
    / SUM(SUM(t.ticket_price)) OVER (), 1
  )                                             AS pct_of_total_revenue
FROM icebase.silver.fact_tickets       t
GROUP BY t.section_tier, t.seat_tier_rank
ORDER BY t.seat_tier_rank DESC;

In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 08: advance_vs_walkup
-- ══════════════════════════════════════════════════════
-- Advance purchase vs day-of revenue and promo behavior.
-- Advance buyers pay more and use fewer promos — important pricing insight.

SELECT
  CASE WHEN t.is_advance_purchase = TRUE THEN 'Advance (>3 days out)'
       ELSE 'Walk-up / Day-of'
  END                                           AS purchase_timing,
  COUNT(*)                                      AS tickets,
  ROUND(AVG(t.ticket_price), 2)                AS avg_ticket_price,
  ROUND(SUM(t.ticket_price), 0)                AS total_revenue,
  ROUND(
    SUM(CASE WHEN t.is_promo_ticket = TRUE THEN 1 ELSE 0 END)
    * 100.0 / COUNT(*), 1
  )                                             AS promo_rate_pct,
  ROUND(AVG(t.days_before_game), 1)            AS avg_days_before_game
FROM icebase.silver.fact_tickets       t
GROUP BY t.is_advance_purchase
ORDER BY avg_ticket_price DESC;